In [1]:
#class TaxiDataExtractor:
    

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("TaxiETL").getOrCreate()

bronze = spark.read.parquet("data/raw/yellow_tripdata_2024-01.parquet")
bronze.printSchema()
bronze.show(5)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------

In [3]:
bronze.select('tpep_pickup_datetime').show()

+--------------------+
|tpep_pickup_datetime|
+--------------------+
| 2024-01-01 00:57:55|
| 2024-01-01 00:03:00|
| 2024-01-01 00:17:06|
| 2024-01-01 00:36:38|
| 2024-01-01 00:46:51|
| 2024-01-01 00:54:08|
| 2024-01-01 00:49:44|
| 2024-01-01 00:30:40|
| 2024-01-01 00:26:01|
| 2024-01-01 00:28:08|
| 2024-01-01 00:35:22|
| 2024-01-01 00:25:00|
| 2024-01-01 00:35:16|
| 2024-01-01 00:43:27|
| 2024-01-01 00:51:53|
| 2024-01-01 00:50:09|
| 2024-01-01 00:41:06|
| 2024-01-01 00:52:09|
| 2024-01-01 00:56:38|
| 2024-01-01 00:32:34|
+--------------------+
only showing top 20 rows



In [7]:
from pyspark.sql import functions as F

bronze_z_czasem = bronze.withColumn(
    "trip_duration_min",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
)


oznaczone = bronze_z_czasem.withColumn(
    "reason",
    F.when(bronze_z_czasem['trip_duration_min'] < 3, "too_short")
     .when(bronze_z_czasem['trip_duration_min'] > 120, "too_long")
     .when(bronze_z_czasem['trip_distance'] <= 0, "bad_distance")
     .when(bronze_z_czasem['fare_amount'] <= 0, "bad_price")
     .otherwise(None)
)

oznaczone.groupBy("reason").count().show()

+------------+-------+
|      reason|  count|
+------------+-------+
|   too_short| 121249|
|   bad_price|  29666|
|        NULL|2777697|
|    too_long|   2742|
|bad_distance|  33270|
+------------+-------+



In [9]:
silver = oznaczone.filter(oznaczone['reason'].isNull())
rejected_basic = oznaczone.filter(oznaczone['reason'].isNotNull())

print("Silver (przed percentylem):", silver.count())
print("Odrzucone (podstawowe warunki):", rejected_basic.count())

Silver (przed percentylem): 2777697
Odrzucone (podstawowe warunki): 186927


In [11]:
percentyl_995 = silver.approxQuantile("fare_amount", [0.995], 0.01)
print(percentyl_995)

[678.5]


In [ ]:
prog_ceny = percentyl_995[0] 

silver_final = silver.filter(silver['fare_amount'] <= prog_ceny)

rejected_high_price = silver.filter(silver['fare_amount'] > prog_ceny) \
    .withColumn("reason", F.lit("high_price"))

In [26]:
rejected_all = rejected_basic.unionByName(rejected_high_price)

print("Silver final:", silver_final.count())
print("Rejected all:", rejected_all.count())

Silver final: 2777697
Rejected all: 186927


In [19]:
silver.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee| trip_duration_min|reason|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+------+
|       2| 2024-01-01 00:57:55|  2024-01-01 01:17:43|              1|         1.72|         1|               

In [24]:
gold = silver_final.withColumn("godzina", F.hour("tpep_pickup_datetime"))

gold = gold.groupBy("godzina", "PULocationID").agg(
    F.count("*").alias("liczba_kursow"),
    F.avg('trip_duration_min').alias("avg_duration_min"),
    F.avg('trip_distance').alias("avg_distance"),
    F.avg('fare_amount').alias("avg_fare"),
    F.sum('trip_duration_min').alias("total_duration_min"),
    F.sum('trip_distance').alias("total_distance"),
    F.sum('fare_amount').alias("total_fare")
)

gold.count()
gold.show(10)

+-------+------------+-------------+------------------+------------------+------------------+------------------+------------------+------------------+
|godzina|PULocationID|liczba_kursow|  avg_duration_min|      avg_distance|          avg_fare|total_duration_min|    total_distance|        total_fare|
+-------+------------+-------------+------------------+------------------+------------------+------------------+------------------+------------------+
|      0|         190|            4|27.483333333333334|              5.04|29.322499999999998|109.93333333333334|             20.16|117.28999999999999|
|      7|         132|         4510| 45.46652254249813|15.422747228381377|62.618070953436835|205054.01666666658| 69556.59000000001| 282407.5000000001|
|      8|         264|          387|14.838544358311795|3.0597674418604655|17.823772609819123| 5742.516666666665|           1184.13| 6897.800000000001|
|     10|         264|          408|14.908782679738561|3.0470833333333336|17.947181372549018| 

In [27]:
print("Gold:", gold.count())

Gold: 5084


In [28]:
rejected_summary = rejected_all.groupBy("reason").count().withColumnRenamed("count", "liczba")
rejected_summary.show()

+------------+------+
|      reason|liczba|
+------------+------+
|   too_short|121249|
|   bad_price| 29666|
|    too_long|  2742|
|bad_distance| 33270|
+------------+------+

